Создайте автокодировщик, удаляющий черные квадраты в случайных областях изображений.

Алгоритм действий:
1. Возьмите базу картинок MNIST.
2. На картинках в случайных местах сделайте чёрные квадраты размера 8 на 8.
3. Создайте и обучите автокодировщик восстанавливать оригинальные изображения из повреждённых изображений.
4. Добейтесь MSE < 0.0070 на тестовой выборке.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision.datasets import MNIST
from torchvision import transforms

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
transform = transforms.ToTensor()

train_data = MNIST(root="./data", train=True, download=True, transform=transform)
test_data = MNIST(root="./data", train=False, download=True, transform=transform)

X_train_org = torch.stack([item[0] for item in train_data]).float()
X_test_org = torch.stack([item[0] for item in test_data]).float()

print("Размер обучающей выборки:", tuple(X_train_org.shape))
print("Размер тестовой выборки:", tuple(X_test_org.shape))

In [ ]:
def add_black_square(images, square_size=8, seed=42):
    rng = np.random.default_rng(seed)
    damaged = images.clone()
    count, channels, height, width = damaged.shape

    for i in range(count):
        y = rng.integers(0, height - square_size + 1)
        x = rng.integers(0, width - square_size + 1)
        damaged[i, :, y:y + square_size, x:x + square_size] = 0.0

    return damaged


X_train_masked = add_black_square(X_train_org, square_size=8, seed=42)
X_test_masked = add_black_square(X_test_org, square_size=8, seed=84)

print("Оригинальные изображения:", tuple(X_train_org.shape))
print("Повреждённые изображения:", tuple(X_train_masked.shape))

In [ ]:
train_dataset = TensorDataset(X_train_masked, X_train_org)
test_dataset = TensorDataset(X_test_masked, X_test_org)

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
class SquareAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),

            nn.Conv2d(64, 1, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


model = SquareAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_count = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            if is_train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        total_count += inputs.size(0)

    return total_loss / total_count


epochs = 10
train_losses = []
test_losses = []

for epoch in range(epochs):
    train_loss = run_epoch(model, train_loader, criterion, optimizer)
    test_loss = run_epoch(model, test_loader, criterion)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    print(f"Эпоха {epoch + 1}/{epochs} | ошибка обучения: {train_loss:.5f} | ошибка проверки: {test_loss:.5f}")

test_mse = test_losses[-1]

print(f"Итоговая ошибка MSE на тестовой выборке: {test_mse:.5f}")

if test_mse < 0.0070:
    print("Условие задания выполнено: MSE меньше 0.0070.")
else:
    print("Условие задания не выполнено: MSE не меньше 0.0070.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Ошибка обучения")
plt.plot(test_losses, label="Ошибка проверки")
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Обучение автокодировщика")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    restored_test_images = model(X_test_masked[:5].to(device)).cpu()

show_count = 5
plt.figure(figsize=(12, 6))

for idx in range(show_count):
    plt.subplot(3, show_count, idx + 1)
    plt.imshow(X_test_masked[idx].squeeze().numpy(), cmap="gray")
    plt.title("Вход")
    plt.axis("off")

    plt.subplot(3, show_count, idx + 1 + show_count)
    plt.imshow(restored_test_images[idx].squeeze().numpy(), cmap="gray")
    plt.title("Восстановлено")
    plt.axis("off")

    plt.subplot(3, show_count, idx + 1 + 2 * show_count)
    plt.imshow(X_test_org[idx].squeeze().numpy(), cmap="gray")
    plt.title("Оригинал")
    plt.axis("off")

plt.suptitle("Удаление чёрных квадратов 8x8 на изображениях MNIST")
plt.tight_layout()
plt.show()